In [8]:
import anthropic
import json
import os

# Use the *name* of the variable, not the secret. Example: export ANTHROPIC_API_KEY=sk-ant-...
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. In a terminal: export ANTHROPIC_API_KEY='your-key' "
        "then restart Jupyter, or use a .env loader. Use Kernel → Change kernel if needed so "
        "this notebook sees the same environment as your shell."
    )

client = anthropic.Anthropic(api_key=api_key)

# Change if your key has access to a different default model.
MODEL = "claude-sonnet-4-20250514"


def _text_from_message(message) -> str:
    parts: list[str] = []
    for block in message.content:
        if getattr(block, "type", None) == "text" and getattr(block, "text", None):
            parts.append(block.text)
    return "".join(parts)


def chat(messages, stop_sequences=None, model=None):
    use_model = model if model is not None else MODEL
    kwargs = dict(
        model=use_model,
        max_tokens=1024,
        messages=messages,
    )
    if stop_sequences:
        kwargs["stop_sequences"] = stop_sequences
    response = client.messages.create(**kwargs)
    return _text_from_message(response)

In [9]:
import json
import re


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})


def _parse_json_object(raw: str) -> dict:
    s = raw.strip()
    for prefix in ("```json", "```JSON", "```"):
        if s.startswith(prefix):
            s = s[len(prefix) :].strip()
    if s.endswith("```"):
        s = s[:-3].strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{[\s\S]*\}", s)
        if not m:
            raise
        return json.loads(m.group(0))


def grade_by_model(test_case, output, model=None):
    eval_prompt = f"""You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {json.dumps(test_case)}
Solution: {json.dumps(output)}

Respond with ONLY a JSON object with these keys:
- "strengths": array of 1-3 key strengths
- "weaknesses": array of 1-3 areas for improvement
- "reasoning": concise explanation
- "score": number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, model=model)
    return _parse_json_object(eval_text)

In [10]:
def run_prompt(test_case, model=None):
    messages = []
    add_user_message(messages, test_case)
    response = chat(messages, model=model)
    return response

In [11]:
def run_test_case(test_case, model=None):
    output = run_prompt(test_case, model=model)
    model_grade = grade_by_model(test_case, output, model=model)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "model": model if model is not None else MODEL,
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [12]:
from statistics import mean


def run_eval(dataset, model=None):
    """Run each test case. Pass model=... to override global MODEL for this run."""
    results = []
    label = model if model is not None else MODEL
    print(f"Model: {label}\n")

    for i, test_case in enumerate(dataset, start=1):
        result = run_test_case(test_case, model=model)
        results.append(result)
        preview = test_case if len(test_case) <= 70 else test_case[:67] + "..."
        print(f"  [{i}/{len(dataset)}] score={result['score']}  |  {preview}")

    average_score = mean(r["score"] for r in results)
    print(f"\nAverage score ({label}): {average_score:.3f}")

    print("\n" + "=" * 64)
    print("PER-PROMPT SCORES")
    print("=" * 64)
    for i, r in enumerate(results, start=1):
        print(f"  Prompt {i:>2}:  score = {r['score']}")
    print("=" * 64 + "\n")
    return results

In [13]:
# define your dataset first
dataset = [
    "Write a function to check if a number is prime",
    "Write a function to reverse a string",
    "Write a function to find the max element in a list"
]

# then call run_eval
results = run_eval(dataset)

Model: claude-sonnet-4-20250514

  [1/3] score=7  |  Write a function to check if a number is prime
  [2/3] score=7  |  Write a function to reverse a string
  [3/3] score=7  |  Write a function to find the max element in a list

Average score (claude-sonnet-4-20250514): 7.000


In [ ]:
# Optional: same dataset, several models — each block prints per-case scores + that model's average.
# Uncomment and adjust model ids to what your API key supports.
#
# for model_id in [
#     "claude-sonnet-4-20250514",
#     "claude-opus-4-20250514",
# ]:
#     print("=" * 60)
#     run_eval(dataset, model=model_id)
#
# Or print again from `results` without re-calling the API:
# for r in results:
#     print(r["model"], r["score"], r["test_case"][:50], sep="  |  ")
